# 01 — PreFlight: Data Intake & Quality Control


In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd()
for candidate in [REPO, REPO.parent, REPO.parent.parent]:
    if (candidate / "src" / "neuro").exists():
        REPO = candidate
        break
sys.path.insert(0, str(REPO / "src"))
import os
os.chdir(REPO)
%matplotlib inline


In [ ]:

import json, psutil
import pandas as pd
import nibabel as nib
from neuro.config import DATA_ROOT, FIGURES_DIR, TR_SEC
from neuro.bids import validate_bids, group_counts
from neuro.spark_utils import get_spark, participants_spark
from neuro import viz

report = validate_bids()
participants = report["participants"]
runs = report["runs"]
print("=== BIDS Pre-flight ===")
for k, v in report.items():
    if k not in ("missing", "runs", "participants"):
        print(f"{k}: {v}")
print("\nGroup counts:")
print(group_counts())
print(f"\nMemory available: {psutil.virtual_memory().available / 1e9:.1f} GB")


In [ ]:

spark = get_spark("BladerunnerNeuro_PreFlight")
spark.sql("SELECT 'Spark ready for scaled preprocessing' AS status").show()
participants_spark(spark, participants).groupBy("group_short").count().show()


In [ ]:

if report["n_runs_available"]:
    sample = runs[runs["bold_exists"]].iloc[0]
    img = nib.load(sample["bold_path"])
    print("Sample:", sample["bold_path"])
    print("Shape:", img.shape, "Zooms:", img.header.get_zooms())
viz.plot_group_demographics(participants)
viz.plot_tr_histogram(runs)
report["missing"].head(10)


In [ ]:

from pathlib import Path
nb_name = Path.cwd().name if False else "01_pre_flight"
!jupyter nbconvert --to html notebooks/01_pre_flight.ipynb --output-dir notebooks/html 2>/dev/null || true
